[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabanib/Clases_clasificacion/blob/main/07_clustering_practico.ipynb#copy=true)


[![Open in Anaconda](https://static.anaconda.cloud/content/a22d04e8445b700f28937ab3231b8cded505d0395c63b7a269696722196d5415)](https://anaconda.com/api/nbserve/launch_notebook?nb_url=https%3A%2F%2Fanaconda.com%2Fapi%2Fprojects%2F6905e11d-e422-418c-ad09-bff2871f164e%2Ffiles%2F07_clustering_practico.ipynb%3Fversion%3Dd208ecd9-7a0f-4698-bde5-cafb52e61689)



# Modelos no supervisados: Clustering


Las técnicas de clustering son variadas y nos permiten determinar patrones de agrupamientos que no son fácilmente observables.

A continuación se va tratar de segmentar clientes de compra online. El dataframe es obtenido de la siguiente dirección: https://archive.ics.uci.edu/dataset/352/online+retail. Aunque de ahí se transformado las variables para dejarlo a nivel cliente.




In [ ]:
import pandas as pd
import sklearn.cluster as cl
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
ventas = pd.read_csv("https://raw.githubusercontent.com/pabanib/dataframes/master/online_retail.csv")
ventas.head()

## Descripción de las variables de `online_retail`

El dataframe reúne información del comportamiento de compra de cada cliente durante el período analizado.

| Columna | Descripción | Tipo de variable |
|---|---|---|
| `CustomerID` | Identificador único del cliente. Aunque aparece como número, se utiliza para identificarlo y no representa una cantidad. | Categórica nominal — identificador. |
| `total_monto` | Importe total de las compras registradas para el cliente durante el período analizado. Representa su aporte a la facturación, no la ganancia que genera. | Cuantitativa continua. |
| `total_cantidades` | Cantidad total de unidades compradas por el cliente. Incluye varias unidades de un mismo producto. | Cuantitativa discreta. |
| `distintos_productos` | Cantidad de productos diferentes comprados por el cliente. Describe la variedad de su compra, independientemente de cuántas unidades adquirió de cada producto. | Cuantitativa discreta. |
| `cantidad_operaciones` | Número de operaciones de compra registradas para el cliente durante el período analizado. Permite describir su frecuencia de compra. | Cuantitativa discreta. |
| `hora_mas_frecuente` | Hora del día en la que el cliente registra compras con mayor frecuencia. Se expresa en formato de 24 horas y tiene un comportamiento cíclico: después de las 23 comienza nuevamente la hora 0. | Temporal cíclica. |
| `repitio_mismo_dia` | Indicador de si el cliente realizó más de una operación de compra en una misma fecha. Toma el valor `True` cuando ocurrió y `False` cuando no ocurrió. | Categórica binaria. |
| `producto_mas_comprado` | Producto identificado como el más comprado por el cliente. El campo contiene el código y la descripción del producto. | Categórica nominal. |
| `Country` | País asociado al cliente en los registros de compra. | Categórica nominal. |

**Diferencia entre unidades, productos y operaciones:** si un cliente realiza dos pedidos y en cada uno compra tres unidades del mismo producto, acumula seis unidades, un producto distinto y dos operaciones.

# Análisis exploratorio de los datos


In [ ]:
ventas.dtypes

#### País del consumidor

In [ ]:
ventas.Country.value_counts().plot(kind = 'bar')

Como la gran mayoría de los clientes son del Reino Unido, vamos convertir esta variable en una dicotómica: 1 si pertenece a UK y 0 en caso contrario

In [ ]:
ventas['UK'] = ventas.Country.apply(lambda x: 1 if x == 'United Kingdom' else 0)

#### Producto más comprado

En cada operación el cliente puede ejecutar la compra de diversos productos. Para unirlos en un cliente se coloca el producto que el cliente más compra.

In [ ]:
ventas.producto_mas_comprado.value_counts()

En este caso, también nos vamos a quedar con los 4 productos más que son los más elegidos por los consumidores. Creamos una variable dummy por cada uno de ellos

In [ ]:
l = ventas.producto_mas_comprado.value_counts()[:4].index
for p in l:
    ventas[p] = ventas.producto_mas_comprado.apply(lambda x: 1 if x == p else 0)

ventas[l]

#### Promedios

Agregamos los promedios de gasto total que tiene el cliente por cada  orden que ejecuta y la cantidad de unidades promedio también.

In [ ]:
ventas['gasto_prom_operacion'] = ventas.total_monto/ventas.cantidad_operaciones
ventas['cant_prom_operacion'] = ventas.total_cantidades/ventas.cantidad_operaciones


In [ ]:
ventas.info()

In [ ]:
ventas.plot(kind= 'scatter', x = 'gasto_prom_operacion', y = 'distintos_productos')

In [ ]:
ventas[['gasto_prom_operacion', 'cant_prom_operacion']].plot(kind='box')

In [ ]:
ventas[['gasto_prom_operacion', 'cant_prom_operacion']].describe()

Ambos promedios muestras datos muy atípicos. Estos se deben a clientes grandes probablemente. Para realizar un mejor clustering conviene elimiar estos extremos para que no afecte a la función.

Todo valor que esté por encima de del percentil 90 lo vamos a considerar extremo.

In [ ]:
gpo_v_ext = ventas.gasto_prom_operacion.quantile([0.90]).values[0]
cpo_v_ext = ventas.cant_prom_operacion.quantile([0.90]).values[0]

ventas2 = ventas[(ventas.gasto_prom_operacion < gpo_v_ext) & (ventas.cant_prom_operacion < cpo_v_ext)]

ventas2[['gasto_prom_operacion', 'cant_prom_operacion']].describe()

In [ ]:
def momento_dia(hora):
    if hora < 6:
        return 'madrugada'
    elif hora < 12:
        return 'mañana'
    elif hora < 18:
        return 'tarde'
    else:
        return 'noche'

ventas2['momento_dia'] = ventas2.hora_mas_frecuente.apply(momento_dia)

### Estandarización y OneHot

Continuamos estandarizando las variables numéricas. Hecho muy importante en clustering sino las variables que tienen valores muy altos de por sí, terminan dominando el agrupamiento. Ej: monto de ventas.

También consideramos las variables categóricas que quedan para convertirlas en variables dummy con OneHotEncoder.

Realizamos la estandarización de los datos y hacemos una limpieza simple como transformar los valores nulos en ceros y eliminar la primer y última columna.


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer



In [ ]:
numericas = ['gasto_prom_operacion', 'cant_prom_operacion','distintos_productos','cantidad_operaciones']
categoricas = ['momento_dia','UK',
       '84077 - WORLD WAR 2 GLIDERS ASSTD DESIGNS',
       '84879 - ASSORTED COLOUR BIRD ORNAMENT',
       '22492 - MINI PAINT SET VINTAGE ',
       '85123A - WHITE HANGING HEART T-LIGHT HOLDER']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numericas),
        ('cat', OneHotEncoder(drop='first'), categoricas)
    ]
)

preprocessor.fit_transform(ventas2).shape

# K - Means en Scikit Learn


La librería `Scikit-learn` cuenta con el método ya programado para trabajar. Tiene diversos parámetros ue modifican los resultados, los principales son:

**n_clusters**: Es el númerod e grupos que queremos formar

**init**: Se elige entre el método "*k-means++*" o entre el método "*random*". Es la forma en que se inicializa el método.

**n_init**: es el número de veces que el algoritmo se ejecuta con diferentes semillas de centroides.

**random_state**: para elegri una semilla que nos permita obtener los mismos resultados luego.

Ahora inicializamos el objeto con los parámetros que elijamos.


In [ ]:
X = preprocessor.fit_transform(ventas2)
km = cl.KMeans(n_clusters= 5, n_init = 1) #creamos el objeto tipo kmeans de sklearn
km.fit(X) #ajustamos con el método fit

Una vez que está ajustado, el objeto contiene el método `predict` o `fit_predict` (antes de ajustar) que devuelve un array con las categorías de grupos.


In [ ]:
km.predict(X)

Otra forma de ver las diferentes categorías que asigno a cada uno de los datos, es utilizando el atributo "labels_" del objeto.


In [ ]:
km.labels_

Las variaciones en los parámetros va generando cambios en las asignaciones de grupos, por lo tanto es importante probar diferentes parametrizaciones y evaluar cual devuelve mejores resultados.


In [ ]:
# Cremos un diccionario en dónde generamos objetos KMeans con distintos parámetros.

n_grupos = 5
km = {'km1': cl.KMeans(n_clusters = n_grupos, init = "random", n_init = 1, random_state = 5464),
      'km2': cl.KMeans(n_clusters = n_grupos, init = "random", n_init = 10, random_state = 5464),
      'km3': cl.KMeans(n_clusters = n_grupos, init = "random", n_init = "auto", random_state = 5464),
      'km4': cl.KMeans(n_clusters = n_grupos, init = "k-means++", n_init = 1, random_state = 5464),
      'km5': cl.KMeans(n_clusters = n_grupos, init = "k-means++", n_init = 10, random_state = 5464)
      }

# Para cada objeto del diccionario ajustamos el objeto con el método fit

for v in km.values():
  v.fit(X)

In [ ]:
fig, ax = plt.subplots(1,5, figsize = (20,8))
j = 0
for v in km.values():
  colores = []
  for i in v.labels_:
    if i == 0:
      colores.append('blue')
    elif i == 1:
      colores.append('green')
    elif i == 2:
      colores.append('red')
    elif i == 3:
      colores.append('orange')
    else:
      colores.append('yellow')
  ventas2.plot(kind = 'scatter', x = 'cant_prom_operacion', y = 'gasto_prom_operacion', c = colores, ax = ax[j])
  ax[j].set_axis_off()
  j += 1

**¿Cómo sabemos que parámetros elegir?**

Como todo algorimto de aprendizaje automático estamos minimizando una función, por lo tanto el algoritmo que tenga el valor mínimo de la función de costo es el que vamos a adoptar como más óptimo.

 La función de costo se conococe como **inercia** o **la suma de los cuadrados de las diferencias intraclusters**

\begin{equation}
WSS = \sum_{k=1}^{K}\sum_{i\in k}||x_i-\mu_k||_2^2
\end{equation}


In [ ]:
for k,v in km.items():
  print("para el modelo {} la inercia fue de {}".format(k, v.inertia_))

# ¿Cómo saber cuantos grupos utilizar?


Uno de los problemas que surge en el aprendizaje no supervisado es que no sabemos cual es la respuesta correcta. Sin embargo de antes de empezar el modelo tenemos que definir parámetros que no sabemos si son los que mejor rinden.

El más importante es el número de grupos en el cual yo quiero agrupar los datos, siguiendo lo planteado anteriormente la forma de elegirlos sería minimizando la inercia. ¿Es esto correcto?

La respuesta es que no porque el $WSS$ se hace $0$ cuando $K=n$ y esto significaría que no ha aprendido nada.

Una de las técnicas utilizadas para definir es el gráfico de codo (**elbow plot**) en estos casos vamos a elegir la cantidad de grupos en los que la mejora en la inercia no sea tan notoria.


![Diagrama de clustering](https://github.com/cristiandarioortegayubro/BDS/blob/main/images/Clustering-001.png?raw=true)


In [ ]:
n_grups = range(2,15)

inercia = []
for i in n_grups:
  km_ = cl.KMeans(n_clusters = i, n_init = 10,init = "k-means++", random_state = 5464 )
  km_.fit(X)
  inercia.append(km_.inertia_)
plt.scatter(n_grups, inercia)
plt.plot(n_grups, inercia)
plt.grid()
#fig = px.line(x=n_grups, y=inercia, title='Inercia KMenas')
#fig.show()

En la práctica el punto de quiebre no es tan claro. Sin embargo, parece que la cantidad de 5 grupos es la óptima en estos casos. 

## Interpretación de resultados

In [ ]:
rdos = ventas2[numericas].groupby(km['km5'].labels_).mean()
rdos = pd.concat([ventas2.groupby(km['km5'].labels_).count()[['CustomerID']],rdos]
                            , axis = 1 )
rdos

In [ ]:
ventas2[categoricas].groupby(km['km5'].labels_).agg(lambda x: x.mode().iloc[0] if not x.empty else None)

In [ ]:
rdos = pd.concat([rdos,ventas2[categoricas].groupby(km['km5'].labels_).
                  agg(lambda x: x.mode().iloc[0] if not x.empty else None)], axis = 1 )

rdos

# Agrupamiento Aglomerativo o Jerárquico


Con la librería `scikitlearn` también tenemos la posibilidad de agrupar los datos de acuerdo a la técnica de árboles jerárquicos.

Dentro de los parámetros que tiene el método, los principales son:

**n_clusters**: Cantidad de grupos

**affinity**: Es la métrica usada para calcular las distancias entre los diferentes datos. Generalmente se usa la euclidiana.

**metric**: Es la métrica que se va usar en el enlace pueden ser varias pero si el enlace es "ward" entonces solo se acepta la euclidiana.

**linkage**: Puede ser "ward", "complete", "average", "single" . Es la forma en que se van a ir conectando los datos.

La forma de trabajo con `Scikit-learn` es la misma que con `K-means`. En general es muy similar a la forma de trabajo que tiene la librería.


In [ ]:
aglo = cl.AgglomerativeClustering(n_clusters = 5)
aglo.fit(X)

In [ ]:
aglo.labels_

In [ ]:
ventas2[numericas].groupby(aglo.labels_).mean()

In [ ]:
n_grupos = 5

aglo = {"aglo1" : cl.AgglomerativeClustering(n_clusters = n_grupos,  metric = 'euclidean', linkage = 'ward'),
        "aglo2" : cl.AgglomerativeClustering(n_clusters = n_grupos,  metric = 'euclidean', linkage = 'single'),
        "aglo3" : cl.AgglomerativeClustering(n_clusters = n_grupos,  metric = 'euclidean', linkage = 'complete'),
        "aglo4" : cl.AgglomerativeClustering(n_clusters = n_grupos,  metric = 'euclidean', linkage = 'average'),
        "aglo5" : cl.AgglomerativeClustering(n_clusters = n_grupos,  metric = 'cosine', linkage = 'average')
    }

for v in aglo.values():
  v.fit(X)

In [ ]:
fig, ax = plt.subplots(1,5, figsize = (20,8))
j = 0
for v in aglo.values():
  colores = []
  for i in v.labels_:
    if i == 0:
      colores.append('blue')
    elif i == 1:
      colores.append('green')
    elif i == 2:
      colores.append('red')
    elif i == 3:
      colores.append('orange')
    else:
      colores.append('yellow')
  ventas2.plot(kind = 'scatter', x = 'cant_prom_operacion', y = 'gasto_prom_operacion', c = colores, ax = ax[j])
  ax[j].set_axis_off()
  j += 1

Considerando las dos variables como para el caso de `K-means` se puede ver que la forma que tiene de agrupar es diferente.

Por lo general este tipo de métodos suele identificar de forma más clara los **outlier** y los separa en grupos diferentes. Esto se debe a que si están muy separados se van a conectar entre ellos primero, pero luego van a tardar en unirse al resto.

 Esto hace que haya muchos grupos con pocas observaciones. Funciona bien con muchos datos y muchos grupos, `K-means` no tiende a funcionar muy bien cuando son muchos grupos.


## Interpretación



In [ ]:
rdos2 = ventas2[numericas].groupby(aglo['aglo1'].labels_).mean()
rdos2 = pd.concat([ventas2.groupby(aglo['aglo1'].labels_).count()[['CustomerID']],rdos2]
                            , axis = 1 )
rdos2

In [ ]:
ventas2[categoricas].groupby(aglo['aglo1'].labels_).agg(lambda x: x.mode().iloc[0])

# Métricas de rendimiento


Una de las dificultades que presenta el aprendizaje no supervisado es que no existe algún método que funcione mejor que otro, además, la modificación de los denominados hiperparámetros afecta sensiblemente el rendimiento del algoritmo empleado. Por lo tanto, se busca comparar varias metodologías que se usan habitualmente para la detección de clústeres, e incluso comparar varias veces la misma metodología, pero variando los hiperparámetros.

Para esto hay que evaluarlos usando métricas de rendimineto.

 Como hemos visto una de las métricas para evaluar el algoritmo es la inercia.  El problema que tiene esta que nos da un valor que no se puede comparar o no se puede medir que tan lejos estamos de lo óptimo.

Una forma de subsanar eso es usando el ratio que surge entre la suma de cuadrados entre clusters y la suma de cuadrados total.

\begin{equation}
RBTSS = \frac{BSS}{TSS}
\end{equation}

De esta manera nos va dar un valor que se encuentra entre 0 y 1 siendo 1 el valor óptimo.

 Esta métrica no se encuentra establecida dentro de la librería `Scikit- learn`, por lo tanto si se quiere utilizar es necesario programarla


In [ ]:
import numpy as np
def dist(x,y):
    x = np.array(x)
    y = np.array(y)
    return np.sum((x-y)**2)

def wss(X):
    X = np.array(X)
    C = np.mean(X,axis = 0)
    filas = X.shape[0]
    rdo = []
    for i in range(filas-1):
        x = X[i,:]
        #r = []
        r = dist(x,C)
        #for j in range(1,filas):
        #    y = X[j,:]
        #    r.append(dist(x,y))
        rdo.append(r)
    return np.sum(rdo)

def SSD(X, grupos):
    X = np.array(X)
    grupos = np.array(grupos)
    g = np.unique(grupos)
    WSS = []
    for i in g:
        x = X[grupos == i]
        WSS.append(wss(x))
    TSS = wss(X)
    BSS = TSS-np.sum(WSS)
    RBTSS = BSS/TSS
    return {'TSS': TSS, 'WSS': np.sum(WSS)/len(WSS), 'BSS': BSS, 'RBTSS':RBTSS}

In [ ]:
SSD(X, aglo['aglo5'].labels_)

De acuerdo al diccionario de metodologías jerárquicas ahora podemos calcular esta métrica para cada una. Y por supuesto también podemos compararlo con el método `k-means` calculado previamente...


In [ ]:
for k, v in aglo.items():
  rbtss = round(SSD(X, v.labels_)['RBTSS'],2)
  print("Para el método {} el RBTSS fue de {}".format(k,rbtss))

Ahora lo calculamos para el modelo `K-means` que habíamos seleccionado previamente.


In [ ]:
SSD(X, km['km5'].labels_)

De acuerdo a está métrica de rendimiento nos conviene seguir eligiendo el modelo `K-Means` con las caracaterísticas del Km5


## Coeficiente de Siluetas


Otra métrica muy utilizada en la literatura es el coeficiete de siluetas. Este tiene ventajas sobre la suma de cuadrados de las desviaciones porque castiga el exceso de grupos.

\begin{equation}
s=\frac{b-a}{max(a,b)}
\end{equation}

siendo $a$ la distancia media entre una observación y todos los elementos de su grupo y $b$ la distancia media entre una observación y todos los elementos de su grupo más próximo. Este coeficiente va de -1 a 1 siendo el valor de 1 el que mejor rendimiento presenta.


In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

rdos = {}

for k,v in km.items():
  sil = silhouette_score(X, v.labels_)
  rdos[k] = sil

for k,v in aglo.items():
  sil = silhouette_score(X, v.labels_)
  rdos[k] = sil

pd.DataFrame(rdos, index = ['sil']).T.sort_values('sil')

El coeficiente de siluetas en realidad se calcula unidad por unidad, esto permite definir  que unidad se encuentra bien asignada al grupo y cual esta mal asignada.

Si el coeficiente es positivo, esto indica que el punto se encuentra más cerca de todos los vecinos de su mismo grupo que del grupo vecino más cercano. Por lo tanto está bien asignado.

Si el coeficiente es 0, indica que la asignación es difusa, porque están a la misma distancia entre este grupo y el vecino.

Si el coeficiente es negativo, directamente indica una mala asignación al grupo.

 Una forma de mirar esto más fácil es con el gráfico de siluetas. Este gráfico nos va permitir observar la cantidad de unidades asignadas a cada grupo, cuantas unidades están bien asignadas y  cuantas están mal asignadas.


In [ ]:
#importamos el mapa de colores
from matplotlib import cm

#Creamos una función que realiza el gráfico de siluetas
def graficoSiluetas(X, met):
  labels = np.unique(met.labels_) # define los niveles de los grupos
  n_clusters = labels.shape[0] #cantidad de grupos
  siluetas = silhouette_samples(X, met.labels_) #aplica el coef. de siluetas  uno a uno
  y_ax_lower, y_ax_upper = 0,0  # el lower define el piso en el que va empezar el grupo y el upper en dónde finaliza
  yticks = []
  for i, c in enumerate(labels):
    c_siluetas = siluetas[met.labels_ == c] #determina los valores del coef. para cada grupo
    c_siluetas.sort()
    y_ax_upper += len(c_siluetas)
    color = cm.jet(float(i)/n_clusters)
    plt.barh(range(y_ax_lower, y_ax_upper),
            c_siluetas,
            height = 1,
            edgecolor = 'none',
            color = color
            )
    yticks.append((y_ax_lower + y_ax_upper)/2)
    y_ax_lower += len(c_siluetas)

  silueta_prom = np.mean(siluetas)
  plt.axvline(silueta_prom, color = 'red', linestyle = "--")
  plt.yticks(yticks, labels+1)
  plt.ylabel('Grupo')
  plt.xlabel('Coeficiente de siluetas')
  plt.show()

In [ ]:
graficoSiluetas(X, aglo['aglo1'])

Para el método jerarquico 1 que armamos podemos ver que el grupo 4 casi no tiene asignada unidades que el grupo 2 y 3 presentan un alto porcentaje de unidades mal asignadas y que el grupo 5 parece ser el que mejor asginación presenta.


In [ ]:
graficoSiluetas(X, km['km4'])

Para el `K-means` con los parámetros de km5 vemos que tenemos muy bien asignados los grupos 2 y 3. El grupo 5 se encuentra con muy pocas unidades y un gran porcentaje mal asignada y altos porcentajes de mal asignación para el grupo 4 y el grupo 1.


# Conclusiones


En este colab nosotros:

- Utilizamos algunos métodos de clustering que ofrece la librería `scikit-learn`.

- Seleccionamos los parámetros óptimos en un algoritmo `K-means` y en un agrupamiento jerárquico.

- Aprendimos una técnica para seleccionar el número óptimo de grupos de un algoritmo.

- Evaluamos los clusters obtenidos utilizando las métricas de rendimiento más comunes que existen actualmente.
